#break_even: calculate the break-even percentage based on odds

A break-even percentage (alternatively called "implied odds") is the
percentage of time a bet must win for you to neither win nor lose money,
when making the bet over time. If someone offers you an even money bet
that a coin flip will land heads, the break-even percentage is 50%. Not
because those are the true odds of winning, but because that's how often
you have to win to exactly break even on an even money bet.

Break-even percentage = Risk / (Risk + Win)

from Miller & Davidow, "The Logic of Sports Betting" (2019)

Inputs:
- odds, the odds offered for the game
- units, what type of odds: American, Decimal or Fractional

Outputs:
- be, the break-even percentage

RTB wrote it, 06 February 2023, post-RYP tutoring of Eva in trig

NOTE on different types of odds:

**American odds** are represented by either a positive number or a negative
number, and the number is always at least 100. The minus sign denotes
that the bet is on a favorite. The plus sign denotes an underdog. When
the bet is plus, the number after the plus sign means how much money you
would win if you bet \$100 on that market. So if you make a bet at +350
odds, it means you will win \$350 for every \$100 you bet. If you bet \$10,
you win \$35. If you bet \$200, you win \$700. And so on. When the bet is
minus (i.e., a favorite) the number after the minus sign means how much
money you would have to bet to win $100 on that market. So if it's -110
that means you have to bet \$110 if you want to win \$100. An even-money
bet would be +100 or -100 or EV.

**Decimal odds** are quoted as a single number, such as 1.9 or 2.5 or 6. What that number means is it's the amount that you'll collect from a winning \$1 bet—including the amount of your bet. So if you bet \$1 at 2.5, you will receive \$2.50 from a winning bet— your original \$1 plus \$1.50 in winnings. Therefore, 2 represents an even money bet. Numbers between 1 and 2 represent favorites. And numbers over 2 represent underdogs.

**Fractional odds**. If you see something like 7/4 odds, it means if you bet 4 (the denominator) you stand to win 7 (the numerator). In American, 7/4 is +175. In decimal it's 2.75. You would represent -110 as 10/11 in
fractional odds. Even-money would be 1/1 odds.

It looks like the relevant parts of the book are available online:
https://issuu.com/sbc.global/docs/the_logic_of_sports_betting_v1_0_5/s/12292345

In [1]:
import numpy as np

def break_even(odds, units):
    """
    Calculate the break-even percentage based on odds.

    A break-even percentage (or "implied odds") is the percentage of time
    a bet must win for you to neither win nor lose money over time.

    Break-even percentage = Risk / (Risk + Win)

    From: Miller & Davidow, "The Logic of Sports Betting" (2019)

    Parameters
    ----------
    odds : array-like or float
        The odds offered for the game.
    units : str
        Type of odds. Must be one of:
        - 'American'
        - 'Decimal'
        - 'Fractional'

    Returns
    -------
    be : float or ndarray
        Break-even percentage, rounded to two decimal places.
    """

    odds = np.array(odds, dtype=float)

    if units == 'American':
        print("Using American odds.")
        be = np.zeros_like(odds)
        L = odds > 0
        be[L] = 100 / (100 + odds[L])
        be[~L] = -odds[~L] / (100 - odds[~L])  # Note sign handling for negative odds

    elif units == 'Decimal':
        print("Using Decimal odds.")
        be = 1 / odds

    elif units == 'Fractional':
        print("Using Fractional odds.")
        be = 1 / (odds + 1)

    else:
        raise ValueError("Units for odds incorrectly specified. Use 'American', 'Decimal', or 'Fractional'.")

    # Convert to percentage
    be = np.round(be * 100, 2)

    # Return a scalar if input was scalar
    if be.size == 1:
        return float(be)
    return be


In [4]:
break_even(-150,'American')

Using American odds.


60.0

#Calculate "the hold" (sports book's margin)

**Example**: Imagine there's a Giants-Dodgers game, and the listed prices
have the odds: Giants +130 and Dodgers -150. This means that the Giants
are the underdog, and a \$100 bet on them, if they win, wins you \$130.
Conversely, the Dodgers are the favorite, so you'd need to bet \$150 on
them in order to win \$100 (if they win). The corresponding break-even
percentages are 43.5% (Giants) and 60% (Dodgers). [This means that, over a
large number of such bets, the Giants would need to win 43.5% of the time
in order for you to break even. Similarly, the Dodgers would need to win
60% of the time--but remember that's why their the favorites.]

Now consider the situation in which you bet the Giants. If you buy the
Giants at 43.5%, the sports book is "buying" the Dodgers at 56.5%. But note that the sports book is "selling" the Dodgers at 60%. It's this gap between what the sports book will buy a bet for and what he will sell it for that is known as "the hold." In this case, it's 60% - 56.5% = 3.5%. This is essentially the built-in profit margin of the book maker. Of course, this assumes equal betting on both sides, which the book maker tries to ensure by "price discovery," a process in which they post an opening line with relatively low betting limits, and then move the line according to how much action they take on either side. If all of the initial action picks the Giants, then they move the line to make them less of an underdog.

NOTE that we could make the calculation in the opposite direction. If you
bet the Dodgers at 60%, it means the sports book is willing to buy the
Giants at 40%, but only willing to sell them to you at a worse break-even
percentage of 43.5%. Again, the difference (= "the hold") is 3.5%.

from pp. 23-4 of Miller & Davidow, "The Logic of Sports Betting" (2019)

In [5]:
import numpy as np

def the_hold(odds, units):
    """
    Calculate the sportsbook's margin ("the hold").

    Parameters
    ----------
    odds : array-like of length 2
        The odds offered for the two teams (e.g., [+130, -150]).
    units : str
        Type of odds: 'American', 'Decimal', or 'Fractional'.

    Returns
    -------
    h : float
        The hold percentage. Positive values favor the bookmaker;
        negative values favor the bettor.

    Notes
    -----
    Based on Miller & Davidow, "The Logic of Sports Betting" (2019).
    Example:
        >>> the_hold([+130, -150], 'American')
        3.5
    """

    # Import or define break_even() beforehand
    #from break_even import break_even  # assumes break_even is in same directory

    # Calculate break-even percentages for each team
    be = break_even(odds, units)

    # Compute the hold: difference in break-even percentages
    # NOTE: It gets more complicated if one allows for ties
    # (called a "push" in sports betting lingo), in which case everyone gets his
    # money back, as if nothing ever happened.
    h = be[1] - (100 - be[0])

    return h


In [7]:
h = the_hold([+130,-150],'American')
print(h)

Using American odds.
3.479999999999997
